In [1]:
%use kandy

In [2]:
import dev.biserman.planet.utils.UtilityExtensions.signPow

In [45]:
class Species(
    val name: String,
    val biomass: Double,
    val baseRate: Double,
    val maxGrowth: Double,
    val maxDeath: Double,
    val carryingCapacity: Pair<String, Double>? = null,
    val interactions: (Species) -> Double?,
)

fun interactionMap(map: Map<String, Double>) = { species: Species -> map[species.name] }


In [35]:
fun <T> (Iterable<T>).toPairs(): Sequence<Pair<T, T>> = sequence {
    val list = this@toPairs.toList()
    for (i in 0 until list.size) {
        for (j in 0 until list.size) {
            yield(list[i] to list[j])
        }
    }
}

In [46]:
import kotlin.random.Random

class Ecosystem(var populations: Map<Species, Int>, var carryingCapacities: Map<String, Int>)

fun tickEcosystem(ecosystem: Ecosystem): Ecosystem {
    return Ecosystem(
        ecosystem.populations.mapValues { (species, population) ->
            val carryingCapacity = ecosystem.carryingCapacities[species.carryingCapacity?.first]
            val carryingCapacityFactor = if (carryingCapacity != null) {
                1 - (ecosystem.populations[species]?.toDouble() ?: 0.0) / carryingCapacity.toDouble()
            } else 1.0

            maxOf(
                0,
                (population + population * (species.baseRate * carryingCapacityFactor + ecosystem.populations.entries.sumOf { (otherSpecies, otherSpeciesPop) ->
                    (species.interactions(
                        otherSpecies
                    ) ?: 0.0) * otherSpeciesPop
                }).coerceIn(species.maxDeath, species.maxGrowth)).roundToInt()
            )
        },
        ecosystem.carryingCapacities
    )
}

In [116]:
val grass = Species(
    name = "grass",
    biomass = 1.0,              // unused currently
    baseRate = 0.60,            // intrinsic growth at low density
    maxGrowth = 0.60,
    maxDeath = -0.75,
    carryingCapacity = "ground" to 0.0,  // .second unused, key is what matters
    interactions = interactionMap(mapOf(
        "hare"  to -0.0000200,   // each hare reduces grass growth
        "mouse" to -0.0000080,
    ))
)

val hare = Species(
    name = "hare",
    biomass = 4.0,
    baseRate = -0.2,           // dies ~10%/tick without food
    maxGrowth = 0.4,
    maxDeath = -0.75,
    carryingCapacity = null,
    interactions = interactionMap(mapOf(
        "grass" to  0.00018,   // energy gained from grazing
        "wolf"  to -0.0045,   // predation loss
        "hawk"  to -0.0001300,
    ))
)

val mouse = Species(
    name = "mouse",
    biomass = 0.03,
    baseRate = -0.2,
    maxGrowth = 0.5,
    maxDeath = -0.75,
    carryingCapacity = null,
    interactions = interactionMap(mapOf(
        "grass" to   +0.00012,
        "hawk"  to-0.0032,
    ))
)

val wolf = Species(
    name = "wolf",
    biomass = 40.0,
    baseRate = -0.1,           // starvation without prey
    maxGrowth = 0.10,           // large animals reproduce slowly
    maxDeath = -0.75,
    carryingCapacity = null,
    interactions = interactionMap(mapOf(
        "hare" to+0.0018,     // ~20% trophic efficiency vs α_hare,wolf
    ))
)

val hawk = Species(
    name = "hawk",
    biomass = 1.0,
    baseRate = -0.1,
    maxGrowth = 0.10,
    maxDeath = -0.75,
    carryingCapacity = null,
    interactions = interactionMap(mapOf(
//        "hare"  to 0.000133,    // ~10% efficiency vs α_hare,hawk
        "mouse" to +0.0014,
    ))
)

val testEcosystem = Ecosystem(
    populations = mapOf(grass to 8000, hare to 150, mouse to 400, wolf to 5, hawk to 5),
    mapOf("carpet" to 10000)
)
val history = (1..500).scan(testEcosystem) { ecosystem, i ->
    val nextEcosystem = tickEcosystem(ecosystem)
    println("=========================")
    println("step $i")
    nextEcosystem.populations.forEach {
        println("${it.key.name}: ${it.value}")
    }
    println("=========================")
    nextEcosystem
}

step 1
grass: 12750
hare: 210
mouse: 600
wolf: 6
hawk: 6
step 2
grass: 20285
hare: 294
mouse: 900
wolf: 7
hawk: 7
step 3
grass: 32191
hare: 412
mouse: 1350
wolf: 8
hawk: 8
step 4
grass: 50893
hare: 577
mouse: 2025
wolf: 9
hawk: 9
step 5
grass: 80017
hare: 808
mouse: 3038
wolf: 10
hawk: 10
step 6
grass: 124789
hare: 1131
mouse: 4557
wolf: 11
hawk: 11
step 7
grass: 192290
hare: 1583
mouse: 6836
wolf: 12
hawk: 12
step 8
grass: 291060
hare: 2216
mouse: 10254
wolf: 13
hawk: 13
step 9
grass: 428920
hare: 3102
mouse: 15381
wolf: 14
hawk: 14
step 10
grass: 606884
hare: 4343
mouse: 23072
wolf: 15
hawk: 15
step 11
grass: 806284
hare: 6080
mouse: 34608
wolf: 17
hawk: 17
step 12
grass: 968779
hare: 8512
mouse: 51912
wolf: 19
hawk: 19
step 13
grass: 982791
hare: 11917
mouse: 77868
wolf: 21
hawk: 21
step 14
grass: 726003
hare: 16684
mouse: 116802
wolf: 23
hawk: 23
step 15
grass: 240963
hare: 23358
mouse: 175203
wolf: 25
hawk: 25
step 16
grass: 60241
hare: 32701
mouse: 262805
wolf: 28
hawk: 28
step 1

In [117]:
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.line
import org.jetbrains.kotlinx.kandy.letsplot.layers.points

val logFn = { x: Double -> log(x, 10.0) }
val identityFn = { x: Double -> x }
val chosenFn = logFn

plot {
    listOf(
        grass to Color.GREEN,
        hare to Color.YELLOW,
        mouse to Color.PURPLE,
        wolf to Color.RED,
        hawk to Color.BLUE
    ).filter { (species, _) -> history.any { it.populations.containsKey(species) } }
        .forEach { (species, lineColor) ->
            line {
                x(0..<history.size, name = "time")
                y(history.map { chosenFn(it.populations[species]!!.toDouble()) }, name = "${species.name}_population")
                color = lineColor
            }
        }
}


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="DGMdM5"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"time",
"y":"grass_population"
},
"stat":"identity",
"data":{
"time":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0,101.0,102.0,103.0,104.0,105.0,106.0,107.0,108.0,109.0,110.0,111.0,112.0,113.0,114.0,115.0,116.0,117.0,118.0,119.0,120.0,121.0,122.0,123.0,124.0,125.0,126.0,127.0,128.0,129.0,130.0,131.0,132.0,133.0,134.0,135.0,136.0,137.0,138.0,139.0,140.0,141.0,142.0,143.0,144.0,145.0,146.0,147.0,148.0,149.0,150.0,151.0,152.0,153.0,154.0,155.0,156.0,157.0,158.0,159.0,160.0,161.0,162.0,163.0,164.0,165.0,166.0,167.0,168.0,169.0,170.0,171.0,172.0,173.0,174.0,175.0,176.0,177.0,178.0,179.0,180.0,181.0,182.0,183.0,184.0,185.0,186.0,187.0,188.0,189.0,190.0,191.0,192.0,193.0,194.0,195.0,196.0,197.0,198.0,199.0,200.0,201.0,202.0,203.0,204.0,205.0,206.0,207.0,208.0,209.0,210.0,211.0,212.0,213.0,214.0,215.0,216.0,217.0,218.0,219.0,220.0,221.0,222.0,223.0,224.0,225.0,226.0,227.0,228.0,229.0,230.0,231.0,232.0,233.0,234.0,235.0,236.0,237.0,238.0,239.0,240.0,241.0,242.0,243.0,244.0,245.0,246.0,247.0,248.0,249.0,250.0,251.0,252.0,253.0,254.0,255.0,256.0,257.0,258.0,259.0,260.0,261.0,262.0,263.0,264.0,265.0,266.0,267.0,268.0,269.0,270.0,271.0,272.0,273.0,274.0,275.0,276.0,277.0,278.0,279.0,280.0,281.0,282.0,283.0,284.0,285.0,286.0,287.0,288.0,289.0,290.0,291.0,292.0,293.0,294.0,295.0,296.0,297.0,298.0,299.0,300.0,301.0,302.0,303.0,304.0,305.0,306.0,307.0,308.0,309.0,310.0,311.0,312.0,313.0,314.0,315.0,316.0,317.0,318.0,319.0,320.0,321.0,322.0,323.0,324.0,325.0,326.0,327.0,328.0,329.0,330.0,331.0,332.0,333.0,334.0,335.0,336.0,337.0,338.0,339.0,340.0,341.0,342.0,343.0,344.0,345.0,346.0,347.0,348.0,349.0,350.0,351.0,352.0,353.0,354.0,355.0,356.0,357.0,358.0,359.0,360.0,361.0,362.0,363.0,364.0,365.0,366.0,367.0,368.0,369.0,370.0,371.0,372.0,373.0,374.0,375.0,376.0,377.0,378.0,379.0,380.0,381.0,382.0,383.0,384.0,385.0,386.0,387.0,388.0,389.0,390.0,391.0,392.0,393.0,394.0,395.0,396.0,397.0,398.0,399.0,400.0,401.0,402.0,403.0,404.0,405.0,406.0,407.0,408.0,409.0,410.0,411.0,412.0,413.0,414.0,415.0,416.0,417.0,418.0,419.0,420.0,421.0,422.0,423.0,424.0,425.0,426.0,427.0,428.0,429.0,430.0,431.0,432.0,433.0,434.0,435.0,436.0,437.0,438.0,439.0,440.0,441.0,442.0,443.0,444.0,445.0,446.0,447.0,448.0,449.0,450.0,451.0,452.0,453.0,454.0,455.0,456.0,457.0,458.0,459.0,460.0,461.0,462.0,463.0,464.0,465.0,466.0,467.0,468.0,469.0,470.0,471.0,472.0,473.0,474.0,475.0,476.0,477.0,478.0,479.0,480.0,481.0,482.0,483.0,484.0,485.0,486.0,487.0,488.0,489.0,490.0,491.0,492.0,493.0,494.0,495.0,496.0,497.0,498.0,499.0,500.0],
"grass_population":[3.9030899869919433,4.105510184769973,4.307175012040345,4.507734468072263,4.706658052072828,4.9

In [16]:
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.line
import org.jetbrains.kotlinx.kandy.letsplot.layers.points

plot {
    line {
        x(1..<history.size, name = "time")
        y((1..<history.size).map { i -> history[i].populations[grass]!! - history[i - 1].populations[grass]!! }, name = "d/dt_grass_population")
        color = Color.GREEN
    }
    line {
        x(1..<history.size, name = "time")
        y((1..<history.size).map { i -> history[i].populations[hare]!! - history[i - 1].populations[hare]!! }, name = "d/dt_hare_population")
        color = Color.YELLOW
    }
    line {
        x(1..<history.size, name = "time")
        y((1..<history.size).map { i -> history[i].populations[wolf]!! - history[i - 1].populations[wolf]!! }, name = "d/dt_wolf_population")
        color = Color.RED
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="87tSsr"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"time",
"y":"d/dt_grass_population"
},
"stat":"identity",
"data":{
"time":[1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0],
"d/dt_grass_population":[2930.198120710399,5022.5532185261045,8260.360079178554,13776.819734543678,21203.694434506564,29626.844436473955,35854.4880481788,31077.510597687535,18048.028696011723,3928.288445787417,-3572.6174331440416,-7877.396217056463,-12025.414417597582,-17101.7304088377,-30420.416733335587,-43557.60916226171,-24116.448582947887,-10537.962095836876,-6248.025735714789,-4176.451559604713,-2987.6394361781167,-2233.535210737924,-1722.2505059743853,-1358.663235790722,-1090.6701593960152,-887.5594436842421,-730.1742285446926,-606.0126461450354,-506.60782560206826,-426.0449507169951,-360.08160192915693,-305.6049877332971,82.77120157456898,880.8487627568963,2112.2818752115604,4092.0604151079024,7466.389268912739,13126.852341409143,21829.017606815174,32855.78920963508,41364.18480482024,38259.33999193441,21678.644295850856,6436.42552370546,1096.6620941641158,149.67769826736185,19.589256477571325,2.548847172758542,0.3313881555804983,0.04308110708370805,0.005600555246928707,7.28072423953563E-4,9.464941103942692E-5,1.2304430129006505E-5,1.599575625732541E-6,2.079468686133623E-7,2.703745849430561E-8,3.521563485264778E-9,4.656612873077393E-10,2.9103830456733704E-11,2.9103830456733704E-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
},
"color":"#3ba272",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"time"
},{
"type":"float",
"column":"d/dt_grass_population"
}]
}
},{
"mapping":{
"x":"time",
"y":"d/dt_hare_population"
},
"stat":"identity",
"data":{
"time":[1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0],
"d/dt_hare_population":[133.32370312027024,156.8535406568451,243.0867873793993,292.9031168404349,427.9197454381706,573.8488997719082,595.7634906678195,958.4673192149144,1163.7329714836942,1720.061675309058,2366.7299828556743,3274.2131313227037,4517.69432390433,6210.202509525843,8586.174227472195,11846.649857942924,-460.2774953195694,-7493.45254864078,-7596.685656989721,-6351.8869679723175,-4993.798007037149,-3847.759418087124,-2914.6460612007595,-2253.6024979814683,-1750.5882669091407,-1477.9445468